In [23]:
%run ./config_api_acto

StatementMeta(, acf55cba-672d-4c77-bfed-5e5d33e58529, 25, Finished, Available, Finished)

# Funções de extração da API Acto Gestão

In [24]:
import requests
import json
import pandas as pd
from datetime import datetime, timezone

TOKEN = TOKEN_SANTOS

HOJE = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")

HEADERS = {
        "Accept": "application/json, text/plain, */*",
        "Authorization": f"Bearer {TOKEN}",
        "User-Agent": "Mozilla/5.0 ...",
        "Content-Type": "application/json",
}


def obter_dados_etapa_atual(TOKEN: str, codigo_catalogo: list) -> pd.DataFrame:

    url = "https://actogestaoapi-gdhrfgdfc8bbe8hs.brazilsouth-01.azurewebsites.net/api/RelatoriosEtapa/ObterTempoEtapaRelatorio"
    payload_etapa = {
        "codCatalogos": codigo_catalogo,
        "dataInicio": "2024-01-01T00:00:00.000Z",
        "dataFim": HOJE,
        "ativo": 1,
    }
    r = requests.post(url, json=payload_etapa, headers=HEADERS, timeout=60)
    r.raise_for_status()
    data = r.json()
    df = pd.DataFrame(data if isinstance(data, list) else data.get("data", []))
    return df


def fetch_tabela(payload_str: str) -> pd.DataFrame:
    """Recebe o JSON do --data-raw como string e devolve um DataFrame com os dados."""
    
    url_dados = "https://actogestaoapi-gdhrfgdfc8bbe8hs.brazilsouth-01.azurewebsites.net/api/Tabela/VisualizarDadosIntermediarios"
    
    config = json.loads(payload_str)
    resp = requests.post(url_dados, headers=HEADERS, json=config)
    print("Status:", resp.status_code)
    
    if resp.status_code != 200:
        print("Corpo da resposta:", resp.text[:500])
        raise SystemExit("❌ Erro ao chamar API")
    
    data = resp.json()
    
    lista_final = []
    if "data" in data and isinstance(data["data"], list):
        for item in data["data"]:
            dados_dict = item.get("dados", {})
            if isinstance(dados_dict, dict):
                for key, lista in dados_dict.items():
                    if isinstance(lista, list):
                        lista_final.extend(lista)
    if not lista_final:
        print("✅ Requisição OK, mas sem linhas.")
        return pd.DataFrame()
    
    df = pd.DataFrame(lista_final)
    print(f"✅ Linhas em pandas: {len(df)}")
    return df


def adicionar_etapa_atual(
    df_etapas: pd.DataFrame, df_solicitacoes: pd.DataFrame
) -> pd.DataFrame:
    temp = df_etapas.groupby("seqFluxo", as_index=False)["dataAtenderEtapa"].max()
    temp_etapa_os = (
        df_etapas.merge(temp, on=["seqFluxo", "dataAtenderEtapa"], how="inner")
        .groupby("seqFluxo", as_index=False)
        .size()
    )
    lista = (
        temp_etapa_os.sort_values("size", ascending=False)
        .query("size > 1")["seqFluxo"]
        .tolist()
    )
    temp = temp.loc[~temp["seqFluxo"].isin(lista)].copy()
    df_etapa_atual = df_etapas.merge(
        temp, on=["seqFluxo", "dataAtenderEtapa"], how="inner"
    )
    df_etapa_comeca_mesmo_tempo = df_etapas.query(
        "seqFluxo in @lista and dataEtapaFim.isna()"
    ).sort_values(["seqFluxo", "dataAtenderEtapa"])
    df_etapa_atual2 = pd.concat([df_etapa_atual, df_etapa_comeca_mesmo_tempo], axis=0)
    df_etapa_atual2 = df_etapa_atual2[["seqFluxo", "etapa", "executor"]].copy()
    df_solicitacoes = df_solicitacoes.rename(columns={"Nº Solicitação|1": "seqFluxo"})
    df_solicitacoes = df_solicitacoes.astype({"seqFluxo": "int64"})
    df_solicitacoes["Data Criação|5"] = pd.to_datetime(
        df_solicitacoes["Data Criação|5"], format="ISO8601"
    )
    df_solicitacoes = df_solicitacoes.merge(df_etapa_atual2, on="seqFluxo", how="left")

    return df_solicitacoes


def adicionar_etapa_atual_2(
    df_etapas: pd.DataFrame, df_solicitacoes: pd.DataFrame
) -> pd.DataFrame:
    temp = df_etapas.groupby("seqFluxo", as_index=False)["dataAtenderEtapa"].max()
    temp_etapa_os = (
        df_etapas.merge(temp, on=["seqFluxo", "dataAtenderEtapa"], how="inner")
        .groupby("seqFluxo", as_index=False)
        .size()
    )
    lista = (
        temp_etapa_os.sort_values("size", ascending=False)
        .query("size > 1")["seqFluxo"]
        .tolist()
    )
    temp = temp.loc[~temp["seqFluxo"].isin(lista)].copy()
    df_etapa_atual = df_etapas.merge(
        temp, on=["seqFluxo", "dataAtenderEtapa"], how="inner"
    )
    df_etapa_comeca_mesmo_tempo = df_etapas.query(
        "seqFluxo in @lista and dataEtapaFim.isna()"
    ).sort_values(["seqFluxo", "dataAtenderEtapa"])
    df_etapa_atual2 = pd.concat([df_etapa_atual, df_etapa_comeca_mesmo_tempo], axis=0)
    df_etapa_atual2 = df_etapa_atual2[["seqFluxo", "etapa", "executor"]].copy()
    df_solicitacoes = df_solicitacoes.rename(columns={"Nº Solicitação": "seqFluxo"})
    df_solicitacoes = df_solicitacoes.astype({"seqFluxo": "int64"})
    df_solicitacoes["Data Criação"] = pd.to_datetime(
        df_solicitacoes["Data Criação"], format="ISO8601"
    )
    df_solicitacoes = df_solicitacoes.merge(df_etapa_atual2, on="seqFluxo", how="left")

    return df_solicitacoes

StatementMeta(, acf55cba-672d-4c77-bfed-5e5d33e58529, 26, Finished, Available, Finished)

# Função para consolidar colunas vindo do Acto Gestão

In [ ]:
def aplicar_bfill(df, coluna: str):

    cols_selecao = df.filter(like=coluna).columns

    df[coluna] = (
        df[cols_selecao].bfill(axis=1).iloc[:, 0]
    )
    df = df.drop(columns=cols_selecao)

    return df

# Funções para replicar tb_os_acto

In [1]:
def harmonizar_nome_bairros(df):
    df['bairro_consolidado'] = df['bairro_consolidado'].str.title()
    df['bairro_consolidado'] = (
        df['bairro_consolidado']
            .str.replace("Radio", "Rádio")
            # .str.replace("Ponta Da Praia", "Ponta da Praia")
            .str.replace("Porta Da Praia", "Ponta Da Praia")
            .str.replace("Mr.", "Morro")
            .str.replace("Mor.", "Morro")
            .str.replace("Morro Monte Serrat", "Monte Serrat")
            .str.replace("Boqueirao", "Boqueirão")
            .str.replace("Pompeia", "Pompéia")
            .str.replace("Porto Da Praia", "Porto Ponta Da Praia")
            .str.replace("Morro Jose Menino", "Morro José Menino")
            .str.replace("Morro Da Caneleira", "Morro Caneleira")
            # .str.replace("Chico De Paula", "Chico de Paula")
            .str.replace("Ilheu Alto", "Ilhéu Alto")
            .str.replace("Vila Matias", "Vila Mathias")
            .str.replace("Iriri Alto", "Iriri")
            .str.replace("Ilha Barnabé", "Barnabe")
    )
    return df


def ajustar_nome_colunas(df):
    # Colunas em letras minúsculas e sem acentos/símbolos
    df.columns = (
        df.columns.str.lower()
        .str.strip()
        .str.replace("  ", " ")
        .str.replace("º", "")
        .str.replace(":", "")
        .str.replace(",", "")  # remove vírgula (corrige erro do Delta)
        .str.replace("ç", "c")
        .str.replace("ã", "a")
        .str.replace("ú", "u")
        .str.replace("ê", "e")
        .str.replace("á", "a")
        .str.replace(r"\s+", "_", regex=True)
    )

    # df = df.loc[:, ~df.columns.duplicated()]
    return df


def aplicar_merge_prazo_bairros(acto):


    def processar_prazo():
        prazo = pd.read_excel(ARQUIVO_AUX, sheet_name="aux_prazo")
        prazo = ajustar_nome_colunas(prazo)
        prazo["servico"] = prazo["servico"].str.capitalize()
        return prazo


    def processar_bairros():
        bairros = pd.read_excel(ARQUIVO_AUX, sheet_name="aux_regionais")
        bairros = ajustar_nome_colunas(bairros)
        bairros["bairro_consolidado"] = bairros["bairro"].str.title()
        bairros = bairros.drop(columns=["bairro"])
        return bairros

    prazo = processar_prazo()
    bairros = processar_bairros()

    # Mescla dados (JOIN)
    acto_prazo = (
        acto
        .merge(prazo, on="servico", how="inner")
        .merge(bairros, on="bairro_consolidado", how="left")
    )
    return acto_prazo


def aplicar_merge_prazo_bairros_ouvidoria(acto):


    def processar_prazo():
        prazo = pd.read_excel(ARQUIVO_AUX, sheet_name="aux_prazo")
        prazo = ajustar_nome_colunas(prazo)
        prazo["servico"] = prazo["servico"].str.capitalize()
        prazo = prazo.rename(columns={"servico": "nome_do_servico_avaliado"})
        return prazo


    def processar_bairros():
        bairros = pd.read_excel(ARQUIVO_AUX, sheet_name="aux_regionais")
        bairros = ajustar_nome_colunas(bairros)
        bairros["bairro_consolidado"] = bairros["bairro"].str.title()
        bairros = bairros.drop(columns=["bairro"])
        return bairros

    prazo = processar_prazo()
    bairros = processar_bairros()

    acto["nome_do_servico_avaliado"] = acto["nome_do_servico_avaliado"].str.capitalize()
    # Mescla dados (JOIN)
    acto_prazo = (
        acto
        .merge(prazo, on="nome_do_servico_avaliado", how="left")
        .merge(bairros, on="bairro_consolidado", how="left")
    )
    return acto_prazo


def tratar_datas_prazos(acto_prazo):

    # Trata datas
    acto_prazo["data_de_solicitacao"] = pd.to_datetime(
        acto_prazo["data_de_solicitacao"], dayfirst=True
    )
    acto_prazo["data_de_finalizacao"] = pd.to_datetime(
        acto_prazo["data_de_finalizacao"], dayfirst=True
    )
    acto_prazo.loc[
        acto_prazo["status"] != "Finalizado",
        "data_de_finalizacao"
    ] = pd.NaT

    # Calcula vencimento do prazo
    acto_prazo["data_de_vencimento_prazo"] = acto_prazo[
        "data_de_solicitacao"
    ] + pd.to_timedelta(acto_prazo["prazo_de_conclusao"], unit="D")

    # Calcula tempo de execução se finalizado
    acto_prazo["tempo_de_execucao_real"] = np.where(
        acto_prazo["status"] == "Finalizado",
        (acto_prazo["data_de_finalizacao"] - acto_prazo["data_de_solicitacao"]).dt.days,
        0,
    )
    acto_prazo["tempo_de_execucao_real"] = (
        acto_prazo["tempo_de_execucao_real"].astype(int)
    )

    # Data de hoje como datetime apenas com data (sem hora)
    data_hoje = pd.Timestamp.now(tz="UTC").normalize()

    # Calcula dias até vencimento
    acto_prazo["dias_ate_vencimento"] = (
        acto_prazo["data_de_vencimento_prazo"] - data_hoje
    ).dt.days

    acto_prazo["dias_ate_vencimento"] = np.where(
        acto_prazo["status"].isin(["Finalizado", "Cancelado"]),
        0,
        acto_prazo["dias_ate_vencimento"]
    )

    acto_prazo['mes_solicitacao'] = acto_prazo['data_de_solicitacao'].dt.month
    acto_prazo['ano_solicitacao'] = acto_prazo['data_de_solicitacao'].dt.year

    acto_prazo['mes_finalizacao'] = acto_prazo['data_de_finalizacao'].dt.month
    acto_prazo['ano_finalizacao'] = acto_prazo['data_de_finalizacao'].dt.year

    acto_prazo['mes_vencimento_prazo'] = acto_prazo['data_de_vencimento_prazo'].dt.month
    acto_prazo['ano_vencimento_prazo'] = acto_prazo['data_de_vencimento_prazo'].dt.year

    # Define status do prazo de execução com 3 opções ajustadas para não exibir "Vence hoje" em OS finalizadas
    # Normalizar as datas para comparar apenas a parte da data (sem horário)
    data_hoje_normalizada = pd.to_datetime(datetime.now().date())
    data_vencimento_normalizada = pd.to_datetime(acto_prazo["data_de_vencimento_prazo"]).dt.date
    data_finalizacao_normalizada = pd.to_datetime(acto_prazo["data_de_finalizacao"]).dt.date

    acto_prazo["status_conclusao_servico"] = np.where(
        # Se o status da OS for "Cancelado", marca como "Cancelado"
        acto_prazo["status"].str.lower() == "cancelado",
        "Cancelado",
        np.where(
            # Se o serviço foi finalizado (tem data de finalização)
            acto_prazo["data_de_finalizacao"].notna(),
            # Verificar se foi entregue dentro ou fora do prazo
            np.where(
                data_finalizacao_normalizada <= data_vencimento_normalizada,
                "Dentro do prazo",  # Entregue dentro do prazo
                "Fora do prazo"     # Entregue fora do prazo
            ),
            # Se não foi finalizado (data_finalizacao é nula) - ainda em andamento
            np.where(
                data_vencimento_normalizada > data_hoje_normalizada.date(),
                "Dentro do prazo",  # Ainda em andamento e dentro do prazo
                np.where(
                    data_vencimento_normalizada == data_hoje_normalizada.date(),
                    "Vence hoje",    # Ainda em andamento e vence hoje
                    "Vencido"        # Ainda em andamento mas vencido
                )
            )
        )
    )

    for date_col in ['data_de_solicitacao', 'data_de_finalizacao', 'data_de_vencimento_prazo']:
        acto_prazo[date_col] = acto_prazo[date_col].dt.date

    return acto_prazo


def tratar_base_final_solicitacoes(acto_prazo):

    # Atribui unidade executora (COPAISA ou região)
    servicos_copaisa = [
        "Corte de grama",
        "Avaliação técnica de árvores",
        "Poda de copa de árvore",
        "Poda de raiz de árvore",
        "Remoção de árvores",
    ]

    if "nome_do_servico_avaliado" in acto_prazo.columns:
        acto_prazo["unidade_executora"] = np.where(
            (acto_prazo["nome_do_servico_avaliado"].isin(servicos_copaisa))
            | (acto_prazo["servico"].isin(servicos_copaisa)),
            "COPAISA",
            np.where(
                acto_prazo["secretaria"].str.contains("CET"),
                acto_prazo["secretaria"],
                acto_prazo["regiao"],
            ),
        )
    else:
        acto_prazo["unidade_executora"] = np.where(
            (acto_prazo["servico"].isin(servicos_copaisa)),
            "COPAISA",
            np.where(
                acto_prazo["secretaria"].str.contains("CET"),
                acto_prazo["secretaria"],
                acto_prazo["regiao"],
            ),
        )

    acto_prazo["unidade_executora"] = np.where(
        acto_prazo["secretaria"].str.contains('SEGOV'),
        "SEALURB",
        acto_prazo["unidade_executora"]
    )

    # Caso ainda reste algum valor em branco, atribui rótulo padrão
    acto_prazo["unidade_executora"] = acto_prazo["unidade_executora"].fillna(
        "Não informado"
    )

    if "responsavel_execucao" in acto_prazo.columns:
        # Define responsavel pela execução dos serviços da SEINFRA
        acto_prazo["responsavel_execucao"] = np.where(
            acto_prazo['etapa_atual'] == 'Execução Terceiro',
            "Empresa terceira",
            acto_prazo["responsavel_execucao"]
        )
        acto_prazo["responsavel_execucao"] = np.where(
            acto_prazo["secretaria"].str.contains("SEINFRA")
            & acto_prazo["status"].isin(["Em atendimento", "Pendente atendimento"])
            & acto_prazo["responsavel_execucao"].isna(),
            "A ser definido",
            acto_prazo["responsavel_execucao"]
        )
        acto_prazo["responsavel_execucao"] = np.where(
            acto_prazo["secretaria"].str.contains("SEINFRA")
            & ~acto_prazo["status"].isin(["Em atendimento", "Pendente atendimento"])
            & acto_prazo["responsavel_execucao"].isna(),
            "Execução própria",
            acto_prazo["responsavel_execucao"]
        )

    return acto_prazo


def remover_registros_teste(acto_prazo):

    lista = [
        "André Ygor Bulata Dos Santos",
        "Matheus De Paula Moura Arsenes",
        "Wagner De Morais Pechim",
        "Teste M",
        "Teste Matheus",
        "Teste Teste",
        "Dialla Araujo Souza",
        "Victor Martins Da Silva",
        "Yago Silva De Jesus",
        "Matheus Santos Alves",
        "Milena Firmiano Lopes",
        "Arleque Sandra Aparecida De Souza",
        "Guilherme Martins Pereira"
    ]

    acto_prazo["solicitante"] = acto_prazo["solicitante"].str.strip().str.title()
    acto_prazo = acto_prazo.loc[~acto_prazo["solicitante"].isin(lista)]

    return acto_prazo

StatementMeta(, , -1, SessionStarting, , SessionStarting)